In [ ]:
# =============================================================================
# Zelle 01 – Projektpfad bestimmen und Ordnerstruktur anlegen
# =============================================================================
# Zweck: Die gesamte Ordnerstruktur wird per Skript erzeugt statt von Hand.
#   Damit ist sie reproduzierbar (z. B. auf dem Medion) und dokumentiert.
# Idempotent: Mehrfaches Ausführen ist unschädlich, vorhandene Ordner
#   bleiben unverändert (exist_ok=True).
# .gitkeep: Git versioniert keine leeren Ordner. Eine leere Platzhalter-Datei
#   sorgt dafür, dass die Struktur später im Repo sichtbar ist.
# =============================================================================

from pathlib import Path
import csv
import sys

# --- Projektwurzel bestimmen -------------------------------------------------
# VS Code startet ein Notebook im Ordner des Notebooks (hier: notebooks/).
# Die Projektwurzel liegt dann eine Ebene darüber. Nicht hart codiert, damit
# derselbe Code auch auf dem Medion läuft.
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

# Sicherheitscheck: Stimmt der Projektname? Verhindert, dass die Struktur
# versehentlich in einem falschen Ordner angelegt wird.
EXPECTED_NAME = "_MyMSIT_AI-Agent_repair-playlist"
assert PROJECT_ROOT.name == EXPECTED_NAME, (
    f"Unerwarteter Projektordner: {PROJECT_ROOT}"
)

# --- Soll-Struktur laut Projektanweisungen v0.3 -------------------------------
ORDNER = [
    "config",
    "data/raw/crates",
    "data/raw/playlists",
    "data/raw/index",
    "data/interim",
    "data/processed",
    "data/gold",
    "notebooks",
    "src/playlist_agent/parsers",
    "src/playlist_agent/agent",
    "app",
    "tests",
    "reports/figures",
    "reports/tables",
    "reports/errors",
    "reports/run_reports",
    "docs",
    "presentation",
    "models",
    "logs",
    "tests/fixtures",
    "src/playlist_agent/agent/prompts",
]

# --- Anlegen und protokollieren ----------------------------------------------
protokoll = []
for rel in ORDNER:
    pfad = PROJECT_ROOT / rel
    status = "vorhanden" if pfad.exists() else "neu"
    pfad.mkdir(parents=True, exist_ok=True)
    (pfad / ".gitkeep").touch(exist_ok=True)
    protokoll.append({"ordner": rel, "status": status})

    
# --- Python-Paketstruktur: __init__.py erzeugen ------------------------------
# Warum: Erst __init__.py macht einen Ordner zu einem regulären Python-Paket.
#   Voraussetzung für "pip install -e ." und saubere Imports aus src/.
PAKETE = [
    "src/playlist_agent",
    "src/playlist_agent/parsers",
    "src/playlist_agent/agent",
]
for rel in PAKETE:
    init = PROJECT_ROOT / rel / "__init__.py"
    status = "vorhanden" if init.exists() else "neu"
    init.touch(exist_ok=True)
    protokoll.append({"ordner": f"{rel}/__init__.py", "status": status})


# --- Ergebnis als CSV speichern (Namensschema <NB>_<Zelle>_<slug>) -----------
csv_pfad = PROJECT_ROOT / "reports/tables/01_01_ordnerstruktur.csv"
with open(csv_pfad, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["ordner", "status"])
    writer.writeheader()
    writer.writerows(protokoll)

# --- Ausgabe -----------------------------------------------------------------
print(f"Python      : {sys.version.split()[0]}  ({sys.executable})")
print(f"Projektwurzel: {PROJECT_ROOT}\n")
for p in protokoll:
    print(f"  [{p['status']:9}] {p['ordner']}")
n_neu = sum(p["status"] == "neu" for p in protokoll)
print(f"\n{len(protokoll)} Ordner geprüft, davon {n_neu} neu angelegt.")
print(f"Protokoll gespeichert: {csv_pfad.relative_to(PROJECT_ROOT)}")

In [ ]:
# =============================================================================
# Zelle 02 – Projektdateien anlegen: .gitignore, pyproject.toml, LICENSE, README
# =============================================================================
# Zweck: Basisdateien reproduzierbar per Skript erzeugen.
# Sicherheit: Vorhandene Dateien werden NIE überschrieben (Status "vorhanden").
#   Änderungen danach direkt in der Datei, nicht durch erneutes Generieren.
# .gitignore-Logik: Inhalte privater Ordner werden ignoriert, die .gitkeep-
#   Platzhalter bleiben erhalten, damit die Ordnerstruktur im Repo sichtbar ist.
# =============================================================================

from datetime import date

COPYRIGHT_HOLDER = "AwaTekoete"  

GITIGNORE = """\
# =============================================================================
# .gitignore – public Repo: echte Daten, Musik und Secrets duerfen NIE hinein
# Prinzip: Ordner-Regeln (Kern) + Dateityp-Regeln (Sicherheitsnetz, falls
# echte Dateien versehentlich ausserhalb von data/ landen)
# =============================================================================

# --- 1. Private Ordner (Inhalt ignoriert, .gitkeep bleibt fuer die Struktur) ---
data/**
!data/**/
!data/**/.gitkeep
models/**
!models/.gitkeep
logs/**
!logs/.gitkeep
reports/run_reports/**
!reports/run_reports/.gitkeep
reports/errors/**
!reports/errors/.gitkeep

# --- 2. MLflow (lokales Tracking, ab Phase 4) ---
mlruns/
mlartifacts/
mlflow.db

# --- 3. Sicherheitsnetz: echte Playlists, Indizes, Musik, Modelle ueberall ---
*.crate
*.m3u
*.m3u8
*.wpl
*.fpl
*musik_index*
*.mp3
*.wav
*.aif
*.aiff
*.flac
*.m4a
*.joblib
*.pkl
# Ausnahme: kleine SYNTHETISCHE Testdaten (public-tauglich, s. Projektanweisung)
!tests/fixtures/**

# --- 4. Secrets (Vorlage .env.example bleibt im Repo) ---
.env
.env.*
!.env.example
*.key
*.pem

# --- 5. Python / Jupyter / Build ---
__pycache__/
*.py[cod]
*.egg-info/
build/
dist/
.ipynb_checkpoints/
.pytest_cache/
.venv/

# --- 6. Betriebssystem-Artefakte (u. a. AppleDouble vom Mac, s. EDA-Vorbefunde) ---
._*
.DS_Store
Thumbs.db
desktop.ini

# --- 7. Editor ---
.vscode/
"""

PYPROJECT = """\
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "playlist-agent"
version = "0.1.0"
description = "Agentic repair and sync of Serato/foobar2000 playlists - feasibility study (MSIT)"
readme = "README.md"
requires-python = ">=3.11"
license = "MIT"
authors = [{ name = "%s" }]
dependencies = []          # wird je Phase ergänzt (mutagen, rapidfuzz, ...)

[project.optional-dependencies]
dev = ["ipykernel", "nbstripout", "pytest"]

[tool.setuptools.packages.find]
where = ["src"]
""" % COPYRIGHT_HOLDER

LICENSE = f"""\
MIT License

Copyright (c) {date.today().year} {COPYRIGHT_HOLDER}

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

README = """\
# Playlist Repair Agent

Feasibility study (MSIT, AI-Agent project): Where does deterministic engineering
suffice, and where do ML, fuzzy matching and LLM agents add measurable value
when repairing and synchronising Serato / foobar2000 playlists?

Status: Phase 0 – Setup

> No real music data is part of this repository. Demos use synthetic data only.
"""

DATEIEN = {
    ".gitignore": GITIGNORE,
    "pyproject.toml": PYPROJECT,
    "LICENSE": LICENSE,
    "README.md": README,
}

for name, inhalt in DATEIEN.items():
    pfad = PROJECT_ROOT / name
    if pfad.exists():
        print(f"  [vorhanden] {name}  (nicht überschrieben)")
    else:
        pfad.write_text(inhalt, encoding="utf-8")
        print(f"  [neu      ] {name}  ({len(inhalt.splitlines())} Zeilen)")

In [ ]:
# =============================================================================
# Zelle 03 – Projektkonfiguration config/config.yaml anlegen
# =============================================================================
# Zweck: Zentrale, versionierte Konfiguration. Code liest Werte von hier,
#   statt sie hart zu codieren (z. B. Zielordner je Laptop).
# Public Repo: KEINE Secrets, KEINE Seriennummer (-> .env, Schritt 0.6b).
# Herkunft jedes Werts ist im Kommentar belegt (Befund oder "offen").
# Sicherheit: Vorhandene Datei wird nicht überschrieben.
# =============================================================================

CONFIG_YAML = """\
# Projektkonfiguration Playlist Repair Agent
# Public: keine Secrets, keine Seriennummern (-> .env)
version: 1

stick:
  # Befund anker_info.txt (06.11.2026): Datentraegername "Lexar", FAT32
  volume_label: "Lexar"
  # Primaeres Erkennungsmerkmal (Projektanweisung Workflow 0): Ordner-Signatur.
  # Befund: alle 5 Ordner existieren auf dem Stick (Angabe Projektinhaber).
  signatur:
    - "Musik"
    - "Playlist"
    - "_ScratchLIVE_/Crates"
    - "Playlist_AsusExpertBook_foobar2000"
    - "Playlist_Medion_foobar2000"
  # Alle Signatur-Ordner muessen vorhanden sein. Lockerung nur mit Begruendung.
  signatur_min_treffer: 5

laptops:
  - name: "asus"
    hostname: "ASUSExpertbook"            # Befund: Ausgabe 'hostname'
    zielordner: "Playlist_AsusExpertBook_foobar2000"
    laufwerk_beobachtet: "D:"             # nur Doku, Logik nutzt Signatur
  - name: "medion"
    hostname: "LTMedionErik"              # Befund: Ausgabe 'hostname' Medion
    zielordner: "Playlist_Medion_foobar2000"
    laufwerk_beobachtet: "E:"             # Befund: Laurent_Garnier.m3u

quellen:
  crates: "_ScratchLIVE_/Crates"          # Serato, Wahrheitsquelle
  mac_playlists: "Playlist"               # Mac-M3U
  musik: "Musik"                          # Audio + .m3u/.wpl im Musikordner

index:
  # Befund Index v2: beobachtete Audio-Endungen. In EDA (Phase 1) verifizieren.
  audio_endungen: [".mp3", ".wav", ".aif"] # Videos (.m4v) bewusst ausgeschlossen
  # Befund Index v2: 151 AppleDouble-Dateien "._*", max. 4096 Byte
  appledouble_praefix: "._"
  appledouble_max_bytes: 4096

matching:
  # Schwellen werden in Phase 4 datenbasiert kalibriert (Precision = 100 %).
  # Bewusst leer: keine Annahmen.
  {}
"""

ziel = PROJECT_ROOT / "config" / "config.yaml"
if ziel.exists():
    print(f"[vorhanden] {ziel.relative_to(PROJECT_ROOT)}  (nicht überschrieben)")
else:
    ziel.write_text(CONFIG_YAML, encoding="utf-8")
    print(f"[neu      ] {ziel.relative_to(PROJECT_ROOT)}  "
          f"({len(CONFIG_YAML.splitlines())} Zeilen)")

In [ ]:
# =============================================================================
# Zelle 04 – .env.example (Vorlage, public) und .env (privat) anlegen
# =============================================================================
# Zweck: Maschinenspezifische und geheime Werte aus Code und Repo heraushalten.
# Regel: Echte Werte NIE in diese Zelle schreiben (Code wird committet).
#   Die Zelle erzeugt nur leere Platzhalter; echte Werte von Hand in .env.
# .env wird von der .gitignore ignoriert (Regel ".env"), .env.example nicht.
# Sicherheit: Vorhandene Dateien werden nicht überschrieben.
# =============================================================================

ENV_TEMPLATE = """\
# Lokale, maschinenspezifische Einstellungen – .env NIE committen.
# Vorlage: .env.example (im Repo). Kopie als .env anlegen und Werte eintragen.

# --- Stick-Verankerung (Bestaetigung zur Ordner-Signatur, s. config.yaml) ---
# Seriennummer laut Windows USBSTOR (Geraete-Manager, Ereignis "konfiguriert")
STICK_SERIAL_USBSTOR=

# --- LLM: lokal (Ollama) ---
# Standard-Adresse eines lokal laufenden Ollama-Servers
OLLAMA_HOST=http://localhost:11434

# --- LLM: Claude-API (nur bei bewusst gewaehlter API-Variante, P8) ---
ANTHROPIC_API_KEY=
"""

for name in [".env.example", ".env"]:
    pfad = PROJECT_ROOT / name
    if pfad.exists():
        print(f"  [vorhanden] {name}  (nicht überschrieben)")
    else:
        pfad.write_text(ENV_TEMPLATE, encoding="utf-8")
        print(f"  [neu      ] {name}  ({len(ENV_TEMPLATE.splitlines())} Zeilen)")

In [ ]:
# =============================================================================
# Zelle 05 – Test viz_config: Store44-Stil, Palette, Namensschema
# =============================================================================
# Zweck: Nachweis, dass (1) das eigene Paket importierbar ist, (2) der Stil
#   greift und (3) figur_speichern falsche Namen abweist.
# Ergebnis-Grafik dient zugleich als Farbreferenz fuer die Praesentation.
# =============================================================================

import matplotlib.pyplot as plt
from playlist_agent.viz_config import FARBEN, stil_aktivieren, figur_speichern

stil_aktivieren()

# --- Palette als Balkendiagramm (Farbreferenz) -------------------------------
namen = list(FARBEN.keys())
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(namen, [1] * len(namen), color=[FARBEN[n] for n in namen],
       edgecolor=FARBEN["text_gedimmt"])
for i, n in enumerate(namen):
    ax.text(i, 0.5, FARBEN[n], ha="center", va="center", fontsize=12,
            color=FARBEN["hintergrund"] if n in ("gold", "blau", "text",
                                                  "text_gedimmt") else FARBEN["text"])
ax.set_title("Store44-Palette (viz_config.FARBEN)")
ax.set_yticks([])
ax.tick_params(axis="x", rotation=20)

pfad = figur_speichern(fig, "01_05_store44_palette",
                       PROJECT_ROOT / "reports" / "figures")
plt.show()
print(f"Gespeichert: {pfad.relative_to(PROJECT_ROOT)}")

# --- Gegenprobe: falscher Name muss abgewiesen werden -------------------------
try:
    figur_speichern(fig, "palette_test", PROJECT_ROOT / "reports" / "figures")
    print("FEHLER: falscher Name wurde akzeptiert")
except ValueError as e:
    print(f"OK, abgewiesen: {e}")
plt.close(fig)

In [ ]:
# =============================================================================
# Zelle 06 – Stick über Ordner-Signatur finden und Quellen inventarisieren
# =============================================================================
# Zweck: Vor dem Snapshot messen, was kopiert wird (Anzahl, Typen, Größe).
# Nur lesend: Auf dem Stick wird nichts verändert (Prinzip P2).
# Verankerung: alle Laufwerke A–Z gegen die Signatur aus config.yaml prüfen
#   (Workflow-Schritt 0; Prototyp für src/playlist_agent/anchor.py).
# Ausgabe-CSV enthält nur Zählwerte je Ordner/Endung, keine Dateinamen
#   -> unbedenklich für das public Repo.
# =============================================================================

import string
from collections import Counter
import yaml
import pandas as pd

cfg = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
signatur = cfg["stick"]["signatur"]
min_treffer = cfg["stick"]["signatur_min_treffer"]

# --- 1. Verankerung: Laufwerke gegen Signatur prüfen -------------------------
kandidaten = []
for lw in string.ascii_uppercase:
    wurzel = Path(f"{lw}:/")
    try:
        if not wurzel.exists():
            continue
        treffer = [s for s in signatur if (wurzel / s).is_dir()]
    except OSError:          # z. B. leeres Kartenlesegerät
        continue
    print(f"  {lw}:  {len(treffer)}/{len(signatur)} Signatur-Ordner")
    if len(treffer) >= min_treffer:
        kandidaten.append(wurzel)

assert len(kandidaten) == 1, f"Erwartet genau 1 Stick, gefunden: {kandidaten}"
STICK = kandidaten[0]
print(f"\nStick gefunden: {STICK}")

# --- 2. Inventar der zu sichernden Quellen -----------------------------------
QUELLEN = {
    "crates": cfg["quellen"]["crates"],
    "mac_playlists": cfg["quellen"]["mac_playlists"],
    "foobar_asus": cfg["laptops"][0]["zielordner"],
    "foobar_medion": cfg["laptops"][1]["zielordner"],
}
PLAYLIST_ENDUNGEN = {".m3u", ".m3u8", ".wpl"}

zeilen = []
for quelle, rel in QUELLEN.items():
    for f in (STICK / rel).rglob("*"):
        if f.is_file():
            zeilen.append({"quelle": quelle, "endung": f.suffix.lower() or "(ohne)",
                           "appledouble": f.name.startswith("._"),
                           "bytes": f.stat().st_size})

# Playlists im Musikordner: nur Playlist-Endungen, nicht die Audiodateien
for f in (STICK / cfg["quellen"]["musik"]).rglob("*"):
    if f.is_file() and f.suffix.lower() in PLAYLIST_ENDUNGEN:
        zeilen.append({"quelle": "musik_playlists", "endung": f.suffix.lower(),
                       "appledouble": f.name.startswith("._"),
                       "bytes": f.stat().st_size})

inventar = (pd.DataFrame(zeilen)
            .groupby(["quelle", "endung", "appledouble"], as_index=False)
            .agg(anzahl=("bytes", "size"), bytes_summe=("bytes", "sum")))

csv_pfad = PROJECT_ROOT / "reports/tables/01_06_snapshot_inventar.csv"
inventar.to_csv(csv_pfad, index=False, encoding="utf-8")

print(inventar.to_string(index=False))
print(f"\nGesamt: {inventar['anzahl'].sum()} Dateien, "
      f"{inventar['bytes_summe'].sum() / 1e6:.1f} MB")
print(f"Gespeichert: {csv_pfad.relative_to(PROJECT_ROOT)}")

In [ ]:
# =============================================================================
# Zelle 07 – Snapshot: Playlists und Crates vom Stick nach data/raw kopieren
# =============================================================================
# Zweck: Getreues, unveränderliches Abbild der Quellen (Prinzip P7: Entwicklung
#   nur auf Kopien). AppleDouble-Dateien werden bewusst mitkopiert (EDA/Linter).
# Nachweis: SHA-256 je Datei von Original und Kopie -> bitgenaue Kopie belegt.
# Schutz: Abbruch, wenn der Zielordner schon Dateien enthält (raw nie überschreiben).
# Datenschutz: Manifest mit Dateinamen liegt in data/raw (privat, gitignored);
#   ins Repo geht nur die Zählstatistik in reports/tables.
# =============================================================================

import hashlib
import shutil
from datetime import datetime

def sha256(pfad: Path) -> str:
    """Prüfsumme blockweise berechnen (speicherschonend)."""
    h = hashlib.sha256()
    with open(pfad, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

RAW = PROJECT_ROOT / "data" / "raw"

# Ziel je Quelle: (Quellordner auf dem Stick, Zielordner in data/raw, Filter)
ZIELE = {
    "crates":          (STICK / cfg["quellen"]["crates"], RAW / "crates", None),
    "mac_playlists":   (STICK / cfg["quellen"]["mac_playlists"], RAW / "playlists" / "mac_playlists", None),
    "foobar_asus":     (STICK / cfg["laptops"][0]["zielordner"], RAW / "playlists" / "foobar_asus", None),
    "foobar_medion":   (STICK / cfg["laptops"][1]["zielordner"], RAW / "playlists" / "foobar_medion", None),
    "musik_playlists": (STICK / cfg["quellen"]["musik"], RAW / "playlists" / "musik_playlists", PLAYLIST_ENDUNGEN),
}

# --- Schutz: data/raw darf außer .gitkeep noch nichts enthalten --------------
vorhanden = [p for p in RAW.rglob("*") if p.is_file() and p.name != ".gitkeep"]
assert not vorhanden, f"data/raw enthält bereits {len(vorhanden)} Dateien – Abbruch."

manifest = []
for quelle, (src_root, dst_root, filter_endungen) in ZIELE.items():
    for src in src_root.rglob("*"):
        if not src.is_file():
            continue
        if filter_endungen and src.suffix.lower() not in filter_endungen:
            continue
        rel = src.relative_to(src_root)
        dst = dst_root / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)          # copy2 erhält Änderungsdatum (wichtig für Sync)
        h_src, h_dst = sha256(src), sha256(dst)
        manifest.append({"quelle": quelle, "pfad_relativ": rel.as_posix(),
                         "bytes": src.stat().st_size, "sha256": h_src,
                         "identisch": h_src == h_dst,
                         "mtime": datetime.fromtimestamp(src.stat().st_mtime).isoformat()})

df_m = pd.DataFrame(manifest)
df_m.to_csv(RAW / "snapshot_manifest.csv", index=False, encoding="utf-8")   # privat

pruefung = (df_m.groupby("quelle", as_index=False)
            .agg(anzahl=("sha256", "size"), identisch=("identisch", "sum"),
                 bytes_summe=("bytes", "sum")))
pruefung["snapshot_zeit"] = datetime.now().isoformat(timespec="seconds")
pruefung.to_csv(PROJECT_ROOT / "reports/tables/01_07_snapshot_pruefung.csv",
                index=False, encoding="utf-8")

print(pruefung.to_string(index=False))
print(f"\nKopiert: {len(df_m)} Dateien | bitgenau identisch: {df_m['identisch'].sum()} "
      f"| abweichend: {(~df_m['identisch']).sum()}")